<img src="titanic.webp" width="700" alt="Titanic">


<span style="color:skyblue; font-size:32px; font-weight:bold"> Titanic Survival Prediction 🚢</span>

---

<span style="color:skyblue; font-size:16px"> Project Overview</span>

This project analyzes the famous **Titanic dataset** to build a machine learning model that predicts whether a passenger survived the disaster. 

The complete pipeline includes:
- **Exploratory Data Analysis (EDA)** — understanding the data structure and patterns
- **Feature Engineering** — extracting `Title` from names, creating `FamilySize` and `IsAlone`, and handling missing values
- **Data Cleaning** — imputing missing `Age` and `Embarked` values, encoding categorical features
- **Model Building** — training a `RandomForestClassifier` with 100 trees
- **Evaluation** — achieving **~84% accuracy** using train/test split and classification metrics

<span style="color:skygreen"> Tools & Libraries:</span> `Python`, `pandas`, `NumPy`, `scikit-learn`, `Jupyter Notebook`


<span style="color:blue; font-weight:bold; font-size:18px">Library Used</span>


In [31]:
import pandas                  as pd
import numpy                   as np
from   sklearn.model_selection import train_test_split
from   sklearn.ensemble        import RandomForestClassifier
from   sklearn.metrics         import accuracy_score, classification_report

<span style="color:blue; font-weight:bold; font-size:18px">Uploading the dataset</span> \
<span style="color:#2ecc71">The Titanic dataset contains passenger information from the ill-fated maiden voyage of the RMS Titanic. It is widely used in machine learning to predict passenger survival based on features such as age, sex, passenger class, and family size.</span> 


In [32]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
print(df.info())
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
(891, 12)


In [33]:
df.head(15)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


<span style="color:blue; font-weight:bold; font-size:18px">Extracting the necessary data from the original data</span>


In [34]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand = False)
df['Title'].value_counts()

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5708\3357574234.py:1: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand = False)


Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Col           2
Mlle          2
Major         2
Ms            1
Mme           1
Don           1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [7]:
rare_titles = ['Dr', 'Rev', 'Col', 'Mlle', 'Major', 'Don', 'Capt', 'the', 'Jonkheer', 'Ms', 'Mme', 'Lady', 'Countess', 'Sir']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64

In [35]:
df['Name_Length'] = df['Name'].apply(len)

In [36]:
df[['Name', 'Title', 'Name_Length']].head()

,Name,Title,Name_Length
0,"Braund, Mr. Owen Harris",Mr,23
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,51
2,"Heikkinen, Miss. Laina",Miss,22
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,44
4,"Allen, Mr. William Henry",Mr,24


<span style="color:blue; font-weight:bold; font-size:18px">Fill in the nan values</span>


In [37]:
sum_null_before = df[['Age', 'Cabin', 'Embarked']].isnull().sum()
print(sum_null_before)

Age         177
Cabin       687
Embarked      2
dtype: int64


In [38]:
df['Age'] = df['Age'].fillna(df.groupby(df['Title'])['Age'].transform('mean').round(1).astype(int))

In [39]:
df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Name_Length
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S,Rev,21
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S,Miss,28
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,21.0,1,2,W./C. 6607,23.45,NaN,S,Miss,40
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C,Mr,21
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q,Mr,19


In [40]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [41]:
df['Has_Cabin'] = df['Cabin'].notnull().astype(int)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Name_Length,Has_Cabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,23,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,51,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,22,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,44,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Rev,21,0
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Miss,28,1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,21.0,1,2,W./C. 6607,23.4500,NaN,S,Miss,40,0
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Mr,21,1


In [42]:
sum_null_After = df[['Age', 'Cabin', 'Embarked']].isnull().sum()
print(sum_null_After)
print('*'*30)
print(df['Has_Cabin'].value_counts())

Age           0
Cabin       687
Embarked      0
dtype: int64
******************************
Has_Cabin
0    687
1    204
Name: count, dtype: int64


In [43]:
df['Sex'] = df['Sex'].map({'male':0, 'female':1})

df = pd.get_dummies(df, columns = ['Embarked', 'Title'], drop_first = True)


In [44]:
print(df[['Sex', 'Embarked_Q', 'Embarked_S']].head())

   Sex  Embarked_Q  Embarked_S
0    0       False        True
1    1       False       False
2    1       False        True
3    1       False        True
4    0       False        True


In [46]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1 

df['IsAlone'] = (df['FamilySize']==1).astype(int)

df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,...,False,False,False,True,False,False,False,False,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,...,False,False,False,False,True,False,False,False,2,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,...,True,False,False,False,False,False,False,False,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,...,False,False,False,False,True,False,False,False,2,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,...,False,False,False,True,False,False,False,False,1,1


<span style="color:blue; font-weight:bold; font-size:18px">Removing irrelevant data</span>


In [47]:
cols_to_drop = ['Ticket', 'Name', 'Cabin', 'PassengerId']

df.drop(columns = cols_to_drop, inplace = True)

print(df.dtypes)
print('*' * 30)
print(df.shape)
print('*' * 30)
print(df.isnull().sum())

Survived            int64
Pclass              int64
Sex                 int64
Age               float64
SibSp               int64
Parch               int64
Fare              float64
Name_Length         int64
Has_Cabin           int64
Embarked_Q           bool
Embarked_S           bool
Title_Col            bool
Title_Countess       bool
Title_Don            bool
Title_Dr             bool
Title_Jonkheer       bool
Title_Lady           bool
Title_Major          bool
Title_Master         bool
Title_Miss           bool
Title_Mlle           bool
Title_Mme            bool
Title_Mr             bool
Title_Mrs            bool
Title_Ms             bool
Title_Rev            bool
Title_Sir            bool
FamilySize          int64
IsAlone             int64
dtype: object
******************************
(891, 29)
******************************
Survived          0
Pclass            0
Sex               0
Age               0
SibSp             0
Parch             0
Fare              0
Name_Length       0


In [48]:
df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Name_Length,Has_Cabin,Embarked_Q,...,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,FamilySize,IsAlone
0,0,3,0,22.0,1,0,7.2500,23,0,False,...,False,False,False,True,False,False,False,False,2,0
1,1,1,1,38.0,1,0,71.2833,51,1,False,...,False,False,False,False,True,False,False,False,2,0
2,1,3,1,26.0,0,0,7.9250,22,0,False,...,True,False,False,False,False,False,False,False,1,1
3,1,1,1,35.0,1,0,53.1000,44,1,False,...,False,False,False,False,True,False,False,False,2,0
4,0,3,0,35.0,0,0,8.0500,24,0,False,...,False,False,False,True,False,False,False,False,1,1


<span style="color:blue; font-weight:bold; font-size:18px">Predict With RandomForrest</span>


In [23]:
X = df.drop('Survived', axis = 1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

model = RandomForestClassifier(n_estimators = 100, random_state = 42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

### Evaluation

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy of Model : {acc*100:.2f}%")


print(classification_report(y_test, y_pred))

Accuracy of Model : 83.80%
              precision    recall  f1-score   support

           0       0.86      0.87      0.86       105
           1       0.81      0.80      0.80        74

    accuracy                           0.84       179
   macro avg       0.83      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179

